In [261]:
# Logan Cheng w/ partner Navpreet Kloy
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics

In [262]:
# load dataset
df = pd.read_csv("bank-additional-full.csv", sep=";")
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [263]:
# display all column names in the dataset
print(df.columns)

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx',
       'cons.conf.idx', 'euribor3m', 'nr.employed', 'y'],
      dtype='object')


In [264]:
# dataset structure overiew including data types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [265]:
# class distribution of prediction
df["y"].value_counts()

,count
y,
no,36548
yes,4640


In [266]:
# view categorical values
for col in df.columns:
    if df[col].dtype == 'object':
        print(col, df[col].unique())

job ['housemaid' 'services' 'admin.' 'blue-collar' 'technician' 'retired'
 'management' 'unemployed' 'self-employed' 'unknown' 'entrepreneur'
 'student']
marital ['married' 'single' 'divorced' 'unknown']
education ['basic.4y' 'high.school' 'basic.6y' 'basic.9y' 'professional.course'
 'unknown' 'university.degree' 'illiterate']
default ['no' 'unknown' 'yes']
housing ['no' 'yes' 'unknown']
loan ['no' 'yes' 'unknown']
contact ['telephone' 'cellular']
month ['may' 'jun' 'jul' 'aug' 'oct' 'nov' 'dec' 'mar' 'apr' 'sep']
day_of_week ['mon' 'tue' 'wed' 'thu' 'fri']
poutcome ['nonexistent' 'failure' 'success']
y ['no' 'yes']


In [267]:
# split dataset in features and target variable
x_data = df[['age','job','marital','education','default','housing','loan',
        'contact','month','day_of_week','campaign','pdays',
        'previous','poutcome']].copy()
y_data = df['y']

In [268]:
# Label encoding to turn categorical values to numerical
from sklearn.preprocessing import LabelEncoder

# encoder for target variable
te = LabelEncoder()

# a separate encoder is created for each categorical column and stored in a dictionary to use encoder user input in making predictions
encoders = {}

for col in x_data.columns:
    if x_data[col].dtype == 'object':
        le = LabelEncoder()
        x_data[col] = le.fit_transform(x_data[col])
        encoders[col] = le

# view encoding mapping
for col, encoder in encoders.items():
    mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
    print(col, mapping)

y_data = te.fit_transform(y_data) # encode target variable

job {'admin.': np.int64(0), 'blue-collar': np.int64(1), 'entrepreneur': np.int64(2), 'housemaid': np.int64(3), 'management': np.int64(4), 'retired': np.int64(5), 'self-employed': np.int64(6), 'services': np.int64(7), 'student': np.int64(8), 'technician': np.int64(9), 'unemployed': np.int64(10), 'unknown': np.int64(11)}
marital {'divorced': np.int64(0), 'married': np.int64(1), 'single': np.int64(2), 'unknown': np.int64(3)}
education {'basic.4y': np.int64(0), 'basic.6y': np.int64(1), 'basic.9y': np.int64(2), 'high.school': np.int64(3), 'illiterate': np.int64(4), 'professional.course': np.int64(5), 'university.degree': np.int64(6), 'unknown': np.int64(7)}
default {'no': np.int64(0), 'unknown': np.int64(1), 'yes': np.int64(2)}
housing {'no': np.int64(0), 'unknown': np.int64(1), 'yes': np.int64(2)}
loan {'no': np.int64(0), 'unknown': np.int64(1), 'yes': np.int64(2)}
contact {'cellular': np.int64(0), 'telephone': np.int64(1)}
month {'apr': np.int64(0), 'aug': np.int64(1), 'dec': np.int64(2),

In [269]:
# view encoded values
x_data

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome
0,56,3,1,0,0,0,0,1,6,1,1,999,0,1
1,57,7,1,3,1,0,0,1,6,1,1,999,0,1
2,37,7,1,3,0,2,0,1,6,1,1,999,0,1
3,40,0,1,1,0,0,0,1,6,1,1,999,0,1
4,56,7,1,3,0,0,2,1,6,1,1,999,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,5,1,5,0,2,0,0,7,0,1,999,0,1
41184,46,1,1,5,0,0,0,0,7,0,1,999,0,1
41185,56,5,1,6,0,2,0,0,7,0,2,999,0,1
41186,44,9,1,5,0,0,0,0,7,0,1,999,0,1


In [270]:
# Split dataset into training and test set
X_train, X_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.3, random_state=1)

In [271]:
# Train a Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train,y_train)

# Calculate the accuracy and assign it to dt_accuracy
dt_accuracy = dt_model.score(X_test, y_test)

print("Decision Tree Test Accuracy:", f"{dt_accuracy * 100:.2f}%")

Decision Tree Test Accuracy: 83.00%


In [272]:
# Train a Random Forest with random feature selection
rf_model = RandomForestClassifier(n_estimators=300, max_features= 'sqrt', random_state=42)

rf_model.fit(X_train, y_train)

# Calculate the accuracy and assign it to rf_accuracy
rf_accuracy = rf_model.score(X_test, y_test)

# Print RF Model Accuracy
print("RF Model Accuracy:", f"{rf_accuracy * 100:.2f}", "%")

RF Model Accuracy: 89.03 %


In [273]:
# Make predictions on the test set
y_pred_rf = rf_model.predict(X_test)

# Calculate precisions on the test set
precision_rf = metrics.precision_score(y_test, y_pred_rf)
recall_rf = metrics.recall_score(y_test, y_pred_rf)
f1_rf = metrics.f1_score(y_test, y_pred_rf)

print("RF Precision:", precision_rf)
print("RF Recall:", recall_rf)
print("RF F1-Score:", f1_rf)

RF Precision: 0.5487179487179488
RF Recall: 0.22717622080679406
RF F1-Score: 0.3213213213213213


In [275]:
# Prepare a provision for a new data entry and its outcome predection
def predict_deposit():
    while True:
        try:
            age = int(input("Enter age: "))
            job = input("Enter job (housemaid, services, admin.,blue-collar, technician, retired, management, unemployed, self-employed, unknown, entrepreneur, student): ")
            marital = input("Enter marital status (married, single, divorced, unknown): ")
            education = input("Enter education (basic.4y, high.school, basic.6y, basic.9y, professional.course, unknown, university.degree, illiterate): ")
            default = input("Has credit in default? (yes/no/unknown): ")
            housing = input("Has housing loan? (yes/no/unknown): ")
            loan = input("Has personal loan? (yes/no/unknown): ")
            contact = input("Contact type (cellular/telephone): ")
            month = input("Last contact month (may, jun, jul, aug, oct, nov, dec, mar, apr, sep): ")
            day_of_week = input("Day of week (mon, tue, wed, thu, fri): ")
            campaign = int(input("Number of contacts in campaign: "))
            pdays = int(input("Days since last contact (-1 if none): "))
            previous = int(input("Number of previous contacts: "))
            poutcome = input("Outcome of previous campaign (success, failure, nonexistent): ")

            # Create dataframe
            new_customer = pd.DataFrame([[age, job, marital, education, default, housing, loan,
                                          contact, month, day_of_week, campaign,
                                          pdays, previous, poutcome]],
                                        columns=['age','job','marital','education','default','housing','loan',
                                                 'contact','month','day_of_week','campaign',
                                                 'pdays','previous','poutcome'])

            # Encode categorical values
            for col in new_customer.columns:
              if col in encoders:
                new_customer[col] = encoders[col].transform(new_customer[col])


            # Prediction
            prediction = rf_model.predict(new_customer)[0]

            if prediction == 1:
                print("Prediction: Customer is likely to subscribe to the deposit (YES).")
            else:
                print("Prediction: Customer is unlikely to subscribe to the deposit (NO).")

            another = input("Enter another customer? (yes/no): ")
            if another.lower() != 'yes':
                break

        except ValueError:
            print("Invalid input. Please enter valid values.")

predict_deposit()

Enter age: 33
Enter job (housemaid, services, admin.,blue-collar, technician, retired, management, unemployed, self-employed, unknown, entrepreneur, student): retired
Enter marital status (married, single, divorced, unknown): married
Enter education (basic.4y, high.school, basic.6y, basic.9y, professional.course, unknown, university.degree, illiterate): basic.6y
Has credit in default? (yes/no/unknown): no
Has housing loan? (yes/no/unknown): no
Has personal loan? (yes/no/unknown): no
Contact type (cellular/telephone): telephone
Last contact month (may, jun, jul, aug, oct, nov, dec, mar, apr, sep): aug
Day of week (mon, tue, wed, thu, fri): thu
Number of contacts in campaign: 1
Days since last contact (-1 if none): 1
Number of previous contacts: 0
Outcome of previous campaign (success, failure, nonexistent): success
job ['admin.' 'blue-collar' 'entrepreneur' 'housemaid' 'management' 'retired'
 'self-employed' 'services' 'student' 'technician' 'unemployed' 'unknown']
marital ['divorced' '

In [276]:
# For the feature importance, show a table of features and its corresponding values
feature_importances= pd.DataFrame({'feature': x_data.columns, 'importance': rf_model.feature_importances_})
feature_importances = feature_importances.sort_values('importance', ascending=False)
feature_importances

,feature,importance
0,age,0.250218
8,month,0.128187
9,day_of_week,0.093120
1,job,0.092692
10,campaign,0.091202
3,education,0.081035
11,pdays,0.062184
13,poutcome,0.049911
2,marital,0.037012
5,housing,0.036013



The code uses the Bank Marketing dataset to build a predictive model that determines if a customer is likely to subscribe to a bank term deposit. First, the dataset is built by using relevant customer and campaign features. Since several of these variables are categorical, we converted them into numerical values using label encoding so that they can be used by machine learning algorithms. The data is then split into training and testing sets, and a Random Forest model is trained to learn patterns between the input features and the target variable. After training, the model's performance is evaluated using precision, recall, and F1-score. The random forest model had an accuracy of 89.03%. However, the model’s performance metric indicates that it is not a strong model for predicting customers who subscribe to the deposit. The model correctly predicted only 23% (recall) of customers who subscribed to the deposit out of all who did subscribe. It also accurately predicts customers that subscribe 55% of the time. Altogether, the low F1-score (0.32) demonstrates how this model has poor performance in predicting customers who subscribe to the deposit. Finally, a prediction function allows users to input new data, which then is processed to determine if the customer is likely to subscribe to the bank's term deposit.

https://www.kaggle.com/datasets/henriqueyamahata/bank-marketing